# QUERY 4
Cuentas que cumplan con el patrón scatter-gather con una sola cuenta de separación,
para cuentas que hayan realizado transferencias en USD hacia 5 cuentas distintas dentro
del período [2022-09-01, 2022-09-05] 

In [ ]:
import pandas as pd

# 1. Cargar datos
RUTA_DATASETS = "../../datasets"
NOMBRE_ARCHIVO_TRANSACCION = "LI-Small_Trans.csv"
# NOMBRE_ARCHIVO_TRANSACCION = "transacciones_sample.csv"
NOMBRE_ARCHIVO_SOLUCION = "q4_solucion.csv"

# Definimos los dtypes básicos para evitar problemas de memoria o strings
transacciones = pd.read_csv(
    f"{RUTA_DATASETS}/{NOMBRE_ARCHIVO_TRANSACCION}",
    dtype={
        "From Bank": "string", 
        "Account": "string",
        "To Bank": "string", 
        "Account.1": "string"
    }
)

# 2. Estandarizar nombres de columnas (si vienen con el formato original)
if "Account" in transacciones.columns and "Account.1" in transacciones.columns:
    transacciones = transacciones.rename(columns={
        "Account": "From Account", 
        "Account.1": "To Account"
    })

# 3. Formateo de fechas y montos
transacciones["Timestamp"] = pd.to_datetime(transacciones["Timestamp"])
transacciones["Amount Paid"] = pd.to_numeric(transacciones["Amount Paid"], errors="coerce")

# 4. Filtro base: USD y sin nulos
df = transacciones[
    (transacciones["Payment Currency"] == "US Dollar") & 
    (transacciones["Amount Paid"].notna())
]

print(f"Transacciones válidas en USD: {len(df)}")

In [ ]:
# Filtramos por el período indicado asegurándonos de incluir el último día completo
df_periodo = df[
    (df["Timestamp"] >= "2022-09-01") & 
    (df["Timestamp"] <= "2022-09-05 23:59:59")
].copy()

# Cuenta = (banco, número de cuenta) — clave compuesta para evitar colisiones entre bancos
df_periodo["From"] = df_periodo["From Bank"] + "|" + df_periodo["From Account"]
df_periodo["To"]   = df_periodo["To Bank"]   + "|" + df_periodo["To Account"]

print(f"Transacciones en el período: {len(df_periodo)}")

In [ ]:
# PASO 1: SCATTER (Dispersión)
# Relaciones únicas A -> B usando claves compuestas (banco|cuenta)
relaciones_ab = df_periodo[["From", "To"]].drop_duplicates()

# Contamos a cuántas cuentas destino distintas transfiere cada cuenta origen
conteo_b_por_a = relaciones_ab.groupby("From")["To"].count()

# Filtramos cuentas A que transfirieron a EXACTAMENTE 5 cuentas distintas
cuentas_a_validas = conteo_b_por_a[conteo_b_por_a == 5].index

scatter_df = relaciones_ab[relaciones_ab["From"].isin(cuentas_a_validas)]


# PASO 2: GATHER (Recolección)
# Todas las relaciones B -> C del período
relaciones_bc = df_periodo[["From", "To"]].drop_duplicates()
relaciones_bc = relaciones_bc.rename(columns={
    "From": "Intermediate", 
    "To": "Final"
})

# Cruzamos: el destino de A (B) debe ser el origen de C
patrones = pd.merge(
    scatter_df, 
    relaciones_bc, 
    left_on="To", 
    right_on="Intermediate",
    how="inner"
)

# Agrupamos por (A, C) y contamos cuántas cuentas intermedias B conectan ambas
conteo_caminos = patrones.groupby(["From", "Final"])["Intermediate"].nunique().reset_index()

# Solo los casos donde las 5 cuentas B convergen en un mismo C
resultado_scatter_gather = conteo_caminos[conteo_caminos["Intermediate"] == 5].copy()

print(f"Se encontraron {len(resultado_scatter_gather)} patrones Scatter-Gather.")

if len(resultado_scatter_gather) > 0:
    # Separar claves compuestas "banco|cuenta" en columnas individuales
    resultado_scatter_gather[["From Bank", "From Account"]] = resultado_scatter_gather["From"].str.split("|", expand=True)
    resultado_scatter_gather[["To Bank", "To Account"]]     = resultado_scatter_gather["Final"].str.split("|", expand=True)
    resultado_scatter_gather = resultado_scatter_gather.rename(columns={"Intermediate": "Amount Transactions"})
    resultado_scatter_gather = resultado_scatter_gather[
        ["From Bank", "From Account", "To Bank", "To Account", "Amount Transactions"]
    ].reset_index(drop=True)
else:
    import pandas as pd
    resultado_scatter_gather = pd.DataFrame(
        columns=["From Bank", "From Account", "To Bank", "To Account", "Amount Transactions"]
    )

print(resultado_scatter_gather)
resultado_scatter_gather.to_csv(f"{NOMBRE_ARCHIVO_SOLUCION}", index=False)
print(f"Resultado guardado en {NOMBRE_ARCHIVO_SOLUCION}")